Across experiments:
- average satisfaction on last day
    - niche / mainstream
    - three conditions
- same for profit

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

In [3]:
sns.set_palette("hls", 8)

In [4]:
### exp1
exp1_category_frequencies = pd.read_csv('data_export_exp1/category_frequencies_505.csv')
exp1_consumer_data = pd.read_csv('data_export_exp1/consumer_data_505.csv')
exp1_customer_recommender_data = pd.read_csv('data_export_exp1/customer_recommender_data_505.csv')
exp1_provider_data = pd.read_csv('data_export_exp1/customer_recommender_data_505.csv')
exp1_recommender_data = pd.read_csv('data_export_exp1/recommender_data_505.csv')

### exp2
exp2_category_frequencies = pd.read_csv('data_export_exp2/category_frequencies_505.csv')
exp2_consumer_data = pd.read_csv('data_export_exp2/consumer_data_505.csv')
exp2_customer_recommender_data = pd.read_csv('data_export_exp2/customer_recommender_data_505.csv')
exp2_provider_data = pd.read_csv('data_export_exp2/provider_data_505.csv')
exp2_recommender_data = pd.read_csv('data_export_exp2/recommender_data_505.csv')

### exp3
exp3_category_frequencies = pd.read_csv('data_export_exp3/category_frequencies_505.csv')
exp3_consumer_data = pd.read_csv('data_export_exp3/consumer_data_505.csv')
exp3_customer_recommender_data = pd.read_csv('data_export_exp3/customer_recommender_data_505.csv')
exp3_provider_data = pd.read_csv('data_export_exp3/provider_data_505.csv')
exp3_recommender_data = pd.read_csv('data_export_exp3/recommender_data_505.csv')

In [5]:
controlled_values = ['Mainstream', 'Niche']

exp1_consumer_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp1_provider_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp1_consumer_data['Type'] = exp1_consumer_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})
exp1_provider_data['Type'] = exp1_provider_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})

exp2_consumer_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp2_provider_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp2_consumer_data['Type'] = exp2_consumer_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})
exp2_provider_data['Type'] = exp2_provider_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})

exp3_consumer_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp3_provider_data.rename(columns={'controlled': 'Type'}, inplace=True)
exp3_consumer_data['Type'] = exp3_consumer_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})
exp3_provider_data['Type'] = exp3_provider_data['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})

KeyError: 'Type'

# Exp1 -- Single Recommender

In [ ]:
# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7, 5), sharex=False, sharey=False)

# Plot 1: User Satisfaction Scores on the Last Day
sns.boxplot(data=exp1_consumer_data[(exp1_consumer_data['cycle'] == 60)],
            x='satisfaction_score', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values,
            ax=axes[0, 0])
axes[0, 0].set_ylabel('Consumer Type')
axes[0, 0].legend().remove()  # Remove legend
axes[0, 0].set_xlim(0, 0.2)   # Set x-axis limit
axes[0, 0].set_xlabel('Consumer Utility')

# Plot 2: Cumulative User Utility
grouped_data = exp1_consumer_data.groupby(['consumer_id', 'recommender_id', 'cycle', 'Type'], as_index=False)['satisfaction_score'].sum()
grouped_data['cumulative_satisfaction'] = grouped_data.groupby(['consumer_id', 'recommender_id'])['satisfaction_score'].cumsum()
sns.lineplot(data=grouped_data,
              x='cycle',
              y='cumulative_satisfaction',
              hue='Type',
              hue_order=controlled_values,
              ax=axes[0, 1])
axes[0, 1].set_ylabel('Cumulative Consumer Utility')
axes[0, 1].set_ylim(0, 6)      # Set y-axis limit
axes[0, 1].set_xlabel('Cycle')

# Plot 3: Provider Utility Distribution
grouped_data = exp1_provider_data.groupby(['provider_id', 'Type'], as_index=False).agg({'profit': 'sum'})
grouped_data.reset_index(drop=True, inplace=True)
sns.stripplot(data=grouped_data,
            x='profit', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values,
            ax=axes[1, 0])
axes[1, 0].legend().remove()  # Remove legend
axes[1, 0].set_ylabel('Provider Type')
axes[1, 0].set_xlabel('Provider Utility')
axes[1, 0].set_xlim(0, 300000)  # Set x-axis limit

# Plot 4: Cumulative Profit
grouped_data = exp1_provider_data.groupby(['provider_id', 'cycle', 'Type'], as_index=False)['profit'].sum()
grouped_data['cumulative_profit'] = grouped_data.groupby(['provider_id'])['profit'].cumsum()
sns.lineplot(data=grouped_data,
              x='cycle',
              y='cumulative_profit',
              hue='Type',
              hue_order=controlled_values,
              errorbar=None, 
              ax=axes[1, 1])
axes[1, 1].set_ylabel('Cumulative Provider Utility')
axes[1, 1].set_ylim(0, 140000)  # Set y-axis limit
axes[1, 1].set_xlabel('Cycle')


# Adjust layout so titles and labels don't overlap
plt.tight_layout()

# Save the figure
plt.savefig('img/exp1_combined_plots_2x2.png')

# Show the figure
plt.show()

# Exp2 -- Two Recommenders - Threshold

In [ ]:
# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7, 5), sharex=False, sharey=False)

# Plot 1: Consumer Satisfaction Scores on the Last Day
sns.boxplot(data=exp2_consumer_data[(exp2_consumer_data['cycle'] == 60)],
            x='satisfaction_score', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values, ax=axes[0, 0])
axes[0, 0].set_ylabel('Consumer Type')
axes[0, 0].legend().remove()  # Remove legend
axes[0, 0].set_xlim(0, 0.2)   # Set x-axis limit
axes[0, 0].set_xlabel('Consumer Utility')
axes[0,0].axvline(x=0.04, linestyle='--')

# Plot 2: Cumulative Satisfaction Scores
grouped_data = exp2_consumer_data.groupby(['consumer_id', 'recommender_id', 'cycle', 'Type'], as_index=False)['satisfaction_score'].sum()
grouped_data['cumulative_satisfaction'] = grouped_data.groupby(['consumer_id', 'recommender_id'])['satisfaction_score'].cumsum()
sns.lineplot(data=grouped_data,
             x='cycle',
             y='cumulative_satisfaction',
             hue='Type',
             hue_order=controlled_values,
             errorbar=None,
             ax=axes[0, 1])
axes[0, 1].set_ylabel('Cumulative Consumer Utility')
axes[0, 1].set_ylim(0, 6)      # Set y-axis limit
axes[0, 1].set_xlabel('Cycle')

# Plot 3: Provider Utility Distribution
grouped_data = exp2_provider_data.groupby(['provider_id', 'Type'], as_index=False).agg({'profit': 'sum'})
grouped_data.reset_index(drop=True, inplace=True)
sns.stripplot(data=grouped_data,
            x='profit', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values,
            ax=axes[1, 0])
axes[1, 0].legend().remove()  # Remove legend
axes[1, 0].set_ylabel('Provider Type')
axes[1, 0].set_xlabel('Provider Utility')
axes[1, 0].set_xlim(0, 300000)  # Set x-axis limit

# Plot 4: Cumulative Utility
grouped_data = exp2_provider_data.groupby(['provider_id', 'cycle', 'Type', 'recommender_id'], as_index=False)['profit'].sum()
grouped_data['cumulative_profit'] = grouped_data.groupby(['provider_id', 'Type', 'recommender_id'])['profit'].cumsum()
sns.lineplot(data=grouped_data,
             x='cycle',
             y='cumulative_profit',
             hue='Type',
             hue_order=controlled_values,
             errorbar=None, 
             ax=axes[1, 1])
axes[1, 1].set_ylabel('Cumulative Provider Utility')
axes[1, 1].set_ylim(0, 140000)  # Set y-axis limit
axes[1, 1].set_xlabel('Cycle')

# Adjust layout so titles and labels don't overlap
plt.tight_layout()

# Save the figure
plt.savefig('img/exp2_combined_plots_2x2.png')

# Show the figure
plt.show()


# Exp3 -- Two Recommenders - UCB

In [ ]:
# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7, 5), sharex=False, sharey=False)

# Plot 1: Consumer Utility on the Last Day
sns.boxplot(data=exp3_consumer_data[(exp3_consumer_data['cycle'] == 60)],
            x='satisfaction_score', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values, ax=axes[0, 0])
axes[0, 0].set_ylabel('Consumer Type')
axes[0, 0].legend().remove()  # Remove legend
axes[0, 0].set_xlim(0, 0.2)   # Set x-axis limit
axes[0, 0].set_xlabel('Consumer Utility')

# Plot 2: Cumulative Satisfaction Scores
grouped_data = exp3_consumer_data.groupby(['consumer_id', 'recommender_id', 'cycle', 'Type'], as_index=False)['satisfaction_score'].sum()
grouped_data['cumulative_satisfaction'] = grouped_data.groupby(['consumer_id', 'recommender_id'])['satisfaction_score'].cumsum()
sns.lineplot(data=grouped_data,
             x='cycle',
             y='cumulative_satisfaction',
             hue='Type',
             hue_order=controlled_values,
             errorbar=None,
             ax=axes[0, 1])
axes[0, 1].set_ylabel('Cumulative Consumer Utility')
axes[0, 1].set_ylim(0, 6)      # Set y-axis limit
axes[0, 1].set_xlabel('Cycle')

# Plot 3: Provider Utility Distribution
grouped_data = exp3_provider_data.groupby(['provider_id', 'Type'], as_index=False).agg({'profit': 'sum'})
grouped_data.reset_index(drop=True, inplace=True)
sns.stripplot(data=grouped_data,
            x='profit', y='Type', hue='Type',
            order=controlled_values, hue_order=controlled_values,
            ax=axes[1, 0])
axes[1, 0].legend().remove()  # Remove legend
axes[1, 0].set_ylabel('Provider Type')
axes[1, 0].set_xlabel('Provider Utility')
axes[1, 0].set_xlim(0, 300000)  # Set x-axis limit

# Plot 4: Cumulative Utility
grouped_data = exp3_provider_data.groupby(['provider_id', 'cycle', 'Type', 'recommender_id'], as_index=False)['profit'].sum()
grouped_data['cumulative_profit'] = grouped_data.groupby(['provider_id', 'Type', 'recommender_id'])['profit'].cumsum()
sns.lineplot(data=grouped_data,
             x='cycle',
             y='cumulative_profit',
             hue='Type',
             hue_order=controlled_values,
             errorbar=None, 
             ax=axes[1, 1])
axes[1, 1].set_ylabel('Cumulative Provider Utility')
axes[1, 1].set_ylim(0, 140000)  # Set y-axis limit
axes[1, 1].set_xlabel('Cycle')

# Adjust layout so titles and labels don't overlap
plt.tight_layout()

# Save the figure
plt.savefig('img/exp3_combined_plots_2x2.png')

# Show the figure
plt.show()


## Utility distribution across the three experiments

In [ ]:
# Load the data for each experiment and seed
seeds = [101, 202, 303, 404, 505]
experiments = [1, 2, 3]

# Dictionaries to store the data
exp1_data = {}
exp2_data = {}
exp3_data = {}

# Loading data for all experiments
for seed in seeds:
    exp1_data[seed] = {
        'consumer': pd.read_csv(f'data_export_exp1/consumer_data_{seed}.csv'),
        'provider': pd.read_csv(f'data_export_exp1/provider_data_{seed}.csv')
    }
    
    exp2_data[seed] = {
        'consumer': pd.read_csv(f'data_export_exp2/consumer_data_{seed}.csv'),
        'provider': pd.read_csv(f'data_export_exp2/provider_data_{seed}.csv')
    }
    
    exp3_data[seed] = {
        'consumer': pd.read_csv(f'data_export_exp3/consumer_data_{seed}.csv'),
        'provider': pd.read_csv(f'data_export_exp3/provider_data_{seed}.csv')
    }

# Rename and map 'controlled' to 'Type'
for seed in seeds:
    for exp_data in [exp1_data, exp2_data, exp3_data]:
        exp_data[seed]['consumer'].rename(columns={'controlled': 'Type'}, inplace=True)
        exp_data[seed]['provider'].rename(columns={'controlled': 'Type'}, inplace=True)
        exp_data[seed]['consumer']['Type'] = exp_data[seed]['consumer']['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})
        exp_data[seed]['provider']['Type'] = exp_data[seed]['provider']['Type'].map({'general': 'Mainstream', 'niche': 'Niche'})

# Aggregate data for consumers
all_grouped_data = []

for exp_num, exp_data in zip(experiments, [exp1_data, exp2_data, exp3_data]):
    for seed in seeds:
        consumer_data = exp_data[seed]['consumer']
        grouped_data = consumer_data[consumer_data['cycle'] == 60][['Type', 'satisfaction_score']].groupby('Type', as_index=False).agg({'satisfaction_score': 'mean'})
        grouped_data.rename(columns={'satisfaction_score': 'mean'}, inplace=True)
        grouped_data['experiment'] = f'exp{exp_num}'
        grouped_data['seed'] = seed
        all_grouped_data.append(grouped_data)

final_consumer_df = pd.concat(all_grouped_data, ignore_index=True)

# Aggregate data for providers
all_grouped_provider_data = []

for exp_num, exp_data in zip(experiments, [exp1_data, exp2_data, exp3_data]):
    for seed in seeds:
        provider_data = exp_data[seed]['provider']
        grouped_data = provider_data.groupby(['provider_id', 'Type', 'recommender_id'], as_index=False)['profit'].sum()
        grouped_data = grouped_data[['Type', 'profit']].groupby('Type', as_index=False).mean()
        grouped_data['experiment'] = f'exp{exp_num}'
        grouped_data['seed'] = seed
        all_grouped_provider_data.append(grouped_data)

final_provider_df = pd.concat(all_grouped_provider_data, ignore_index=True)

# Plotting
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Plot 1: Mean Consumer Utility
final_consumer_df['experiment'] = final_consumer_df['experiment'].map({'exp1': 'Single Recommender', 
                                                                       'exp2': 'Threshold Switching',
                                                                      'exp3': 'UCB Switching'})
sns.barplot(data=final_consumer_df, x='experiment', y='mean', hue='Type', ax=axes[0])
axes[0].set_title('Mean Consumer Utility by Interest Type and Experiment')
axes[0].set_xlabel('Experiment')
axes[0].set_ylabel('Mean Consumer Utility')
handles, labels = axes[0].get_legend_handles_labels()
# Update labels while preserving colors
axes[0].legend(handles, ['Mainstream', 'Niche'], title='Consumer Type')


# Plot 2: Mean Provider Profit
final_provider_df['experiment'] = final_provider_df['experiment'].map({'exp1': 'Single Recommender', 
                                                                       'exp2': 'Threshold Switching',
                                                                      'exp3': 'UCB Switching'})
sns.barplot(data=final_provider_df, x='experiment', y='profit', hue='Type', ax=axes[1])
axes[1].set_title('Mean Provider Utility by Content Type and Experiment')
axes[1].set_xlabel('Experiment')
axes[1].set_ylabel('Mean Provider Utility')
# Get original legend handles and labels
handles, labels = axes[1].get_legend_handles_labels()
# Update labels while preserving colors
axes[1].legend(handles, ['Mainstream', 'Niche'], title='Provider Type')

plt.tight_layout()
plt.savefig('img/final_img.png')
plt.show()


In [ ]:
print(exp1_provider_data.columns)


In [ ]:
print(exp1_consumer_data.head())
print(exp1_provider_data.head())
